# x402 endpoint readiness, week by week

A reproducible read of the scvd.store corpus: one signed round per week, one row per host in the public x402 discovery list, with what a single conformance probe saw. Concept DOI: https://doi.org/10.5281/zenodo.22284887. Live copy: https://scvd.store/corpus.json. Licence CC BY 4.0.

Every number below is printed with its denominator. Nothing here is a ranking of one host against another: `ready` means one probe, at one moment, saw a well-formed 402 with a payable offer; `not_ready` names which checks failed; `unreachable` and `not_probed` are counted separately and never folded into either.

Dependencies: the standard library only. `matplotlib` is used if present.

In [ ]:
import json, urllib.request, collections

# Read from the Hugging Face dataset repo by default; switch to the live site for the newest round.
SOURCE = "https://huggingface.co/datasets/keeper-scvd/x402-endpoint-readiness/resolve/main"
# SOURCE = "https://scvd.store"  # live: corpus.json indexes /corpus/{n}.json

def fetch(path):
    with urllib.request.urlopen(f"{SOURCE}/{path}", timeout=60) as r:
        return json.load(r)

index = fetch("corpus.json")
print(index["name"]); print("licence:", index["license"]); print("rounds:", len(index["distribution"]) - 1)

In [ ]:
rounds = []
for entry in index["distribution"]:
    url = entry["contentUrl"]
    if not url.rstrip("/").split("/")[-1].split(".")[0].isdigit():
        continue  # the index itself
    n = url.rstrip("/").split("/")[-1]
    rounds.append(fetch(n))
rounds.sort(key=lambda r: r["snapshot"]["sequence"])
print([r["snapshot"]["week"] for r in rounds])

In [ ]:
def readiness(round_doc):
    r = round_doc["snapshot"]["round"]
    hosts = r["hosts"]
    verdicts = collections.Counter(h["verdict"] for h in hosts)
    walked = len(hosts)
    probed = walked - verdicts.get("not_probed", 0)
    ready = verdicts.get("ready", 0)
    return {
        "week": r["week"],
        "walked": walked,
        "probed": probed,
        "ready": ready,
        "not_ready": verdicts.get("not_ready", 0),
        "unreachable": verdicts.get("unreachable", 0),
        "not_probed": verdicts.get("not_probed", 0),
        "ready_of_probed_pct": round(100 * ready / probed, 1) if probed else None,
        "population_known": (r.get("population") or {}).get("population_known"),
        "coverage_pct": (r.get("population") or {}).get("coverage_pct"),
    }

rows = [readiness(r) for r in rounds]
print(f"{'week':10} {'ready':>6} {'probed':>7} {'walked':>7} {'ready/probed':>13} {'known':>7} {'coverage':>9}")
for x in rows:
    print(f"{x['week']:10} {x['ready']:6d} {x['probed']:7d} {x['walked']:7d} {str(x['ready_of_probed_pct'])+'%':>13} {str(x['population_known']):>7} {str(x['coverage_pct'])+'%':>9}")

In [ ]:
# The named failing checks, per week, with the denominator (hosts that were probed and not ready).
for r in rounds:
    hosts = r["snapshot"]["round"]["hosts"]
    not_ready = [h for h in hosts if h["verdict"] == "not_ready"]
    failed = collections.Counter(c for h in not_ready for c in (h.get("failed") or []))
    print(r["snapshot"]["week"], f"not_ready={len(not_ready)}")
    for check, n in failed.most_common(5):
        print(f"   {check:28} {n:5d} of {len(not_ready)}  ({100*n/len(not_ready):.1f}%)")

In [ ]:
try:
    import matplotlib.pyplot as plt
    weeks = [x["week"] for x in rows]
    plt.figure(figsize=(8, 3.5))
    plt.plot(weeks, [x["ready"] for x in rows], marker="o", label="ready")
    plt.plot(weeks, [x["probed"] for x in rows], marker="o", label="probed (denominator)")
    plt.title("x402 doors ready, of doors probed, per signed round")
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
except ImportError:
    print("matplotlib not installed; the table above is the result.")

## What this does not show

- Whether a door delivers after payment. The probe reads the 402 and the signed offer; it does not pay.
- Anything between rounds. One probe per host per week, at indexer cadence.
- Hosts no directory lists. `population_known` is the union of the sources read that week, never the universe; `coverage_pct` is the share of it that was walked.

Each round file carries its own `digest`, `signature`, `public_key` and `ots` proof. To verify one: recompute the sha256 of the snapshot's canonical JSON, check the ed25519 signature against the key at https://scvd.store/.well-known/scvd-signing-key, and run `ots verify` on the proof. The exact steps are printed in `corpus.json` under `how_to_verify`.